# Chapter 14: the first project, a DO chart with CUSUM
The winning use case is a chart on the lager bright tank release DO. The notebook loads the synthetic week to show where the DO results sit, then a longer synthetic series of 305 release DO results (about 15 months, seed 14): a stable run of 60 sets the limits on log DO. The CUSUM sits beneath.

**No code tier.** Export 30 days of DO results to a sheet. Draw them in time order with the 50 ppb limit. Mark the days you remember something going wrong.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"

In [ ]:
week = pd.read_csv(DATA + "week_result.csv").merge(pd.read_csv(DATA + "week_sample.csv"), on="sample_id")
print(week[week.test_code == "DO"].groupby("sample_point").size().rename("DO results in the week"))
print("one week holds too few release results for a baseline of 60, so the chart uses the longer series below")

In [ ]:
do = pd.read_csv(DATA + "ch14_do_release.csv")
y = np.log(do.do_ppb.to_numpy())
base = y[:60]; cl = base.mean(); s = np.abs(np.diff(base)).mean() / 1.128
k, h = 0.5 * s, 5 * s
C, c = [], 0.0
for v in y: c = max(0.0, c + v - (cl + k)); C.append(c)
C = np.array(C)
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(do.result, do.do_ppb, lw=0.8, color="#2f7f72"); ax[0].axhline(50, color="#b23a2e", label="limit 50 ppb")
for v, st in ((cl, "-"), (cl + 3 * s, "--"), (cl - 3 * s, "--")): ax[0].axhline(math.exp(v), ls=st, color="#0b4f45")
ax[0].set_yscale("log"); ax[0].set_ylabel("DO (ppb, log scale)"); ax[0].legend(fontsize=8)
ax[1].plot(do.result, C, color="#0b4f45"); ax[1].axhline(h, ls="--", color="#b23a2e"); ax[1].set_ylabel("CUSUM on log DO"); ax[1].set_xlabel("release result")
plt.show()

In [ ]:
first_cusum = do.result[np.argmax(C > h)] if (C > h).any() else None
first_limit = do.result[np.argmax(do.do_ppb > 50)] if (do.do_ppb > 50).any() else None
beyond = do.result[np.abs(y - cl) > 3 * s].tolist()
print("first CUSUM alarm at result", first_cusum, "| first result over 50 ppb at", first_limit, "| points beyond the log limits:", beyond)

**What to look at.** The replay question from 14.3: did the chart see the seal wearing before the 50 ppb limit did? By how many results? Count every flag, true or false, in the log.